#### Теория: Запросы

Как правило, запросы в базу формируются динамически на основе данных, которые поступают извне. Например, создание пользователя чаще всего происходит во время регистрации. В таком случае код вставки выглядит так:

```python
import psycopg

name = "Tota"
phone = "1234"

sql = f"INSERT INTO users (username, phone) VALUES ('{name}', '{phone}');"
cursor.execute(sql)
conn.commit()
```

Запросы, составленные таким образом, несут в себе серьезную опасность, называемую `SQL-инъекцией`. Вместо обычных данных, пользователь может отправить текст, изменяющий `SQL-запрос` на тот, который нужен пользователю. 
Представьте, что внутри переменной бы хранилась строка 

```'); INSERT INTO users (username, phone) VALUES ('i am hacker', '777777')```. 

- `')` — это закрывающая скобка и одинарная кавычка. Она «досрочно» закрывает значение поля username в исходном запросе. То есть база «думает», что значение имени уже закончилось.

- `;` — это разделитель команд. Он говорит базе: «предыдущая команда завершена, дальше идёт новая».

- `INSERT INTO users ...` — это уже новая SQL‑команда, которую злоумышленник вставляет сам.

В результате база выполняет и исходный `INSERT`, и дополнительный, который добавил хакер. Так и работает классическая `SQL‑инъекция` через конкатенацию строк.

В таком случае, после подстановки мы бы получили следующий запрос:

```sql
INSERT INTO users (username, phone) VALUES ('somename', '');
INSERT INTO users (username, phone) VALUES ('i am hacker', '777777');
```

##### `Почему это опасно`

В этом примере хакер просто добавляет ещё одну запись в таблицу. Но в реальных сценариях через такую уязвимость можно:

- удалить данные `(DELETE FROM ...)`;
- изменить пароли или роли пользователей;
- прочитать чужие данные `(SELECT * FROM ...)`;
- в некоторых конфигурациях даже выполнять команды на сервере.

Фактически вместо одного запроса получилось два. Таким образом, опытный взломщик может выполнить произвольный запрос в базу данных удалив, добавив или изменив все что ему нужно.

В разработке существует правило - никогда не доверяй пользовательским данным. В случае работы с базой данных, это особенно важно. Любые данные перед вставкой должны быть экранированы. Для этого используются `плейсхолдеры`.

`Плейсхолдер` — это метка в тексте `SQL-запроса`, которая обозначает место для значения. Само значение передаётся отдельным аргументом в `execute()`, а не вставляется прямо в строку. 

`Psycopg` принимает это значение, экранирует его и только потом отправляет запрос в базу. Данные и структура запроса никогда не смешиваются, поэтому `SQL-инъекция` через плейсхолдеры невозможна.

```python
name = "John"
age = 19

with conn.cursor() as curs:
    # для позиционных аргументов всегда передается последовательность, даже если параметр один
    # здесь передается кортеж (name,)
    curs.execute("SELECT id, name FROM users WHERE name=%s;", (name,))
    curs.fetchall()
    conn.commit()
conn.close()
```

Можно использовать именнованые аргументы.
Именованные аргументы в `execute()` — это способ передавать параметры в `SQL‑запрос` не по порядку (как в позиционных), а по именам. В `psycopg`  для этого используют синтаксис` %(имя)s` в запросе и словарь с данными.

```python 
name = "John"
age = 19
with conn.cursor() as curs:
    curs.execute(
        "INSERT INTO users (name, age) VALUES (%(name)s, %(age)s);",
        {"age": age, "name": name},
    )
    conn.commit()

conn.close()
```
- В строке запроса `%(name)s` и `%(age)s` — это «места», куда драйвер подставит значения.

- Словарь `{"age": age, "name": name}` связывает имена из запроса с реальными переменными.

- Драйвер сам подставляет значения, экранирует спецсимволы и защищает от `SQL‑инъекций` — поэтому так делать безопасно, в отличие от `f‑строк`.

- Позиционный плейсхолдер `%s` указывает, что на это место нужно подставить очередное значение из переданного кортежа. 

- Порядок значений в кортеже должен совпадать с порядком плейсхолдеров в запросе. 

- Тип данных на синтаксис не влияет: `%s` используется и для строк, и для чисел, и для дат — `psycopg` сам преобразует `Python`-тип в подходящий  `SQL-литерал`.

- Плейсхолдер не нужно заключать в кавычки.

- Именованный плейсхолдер `%(name)s` ссылается на ключ словаря, который передаётся вторым аргументом. Порядок ключей в словаре не важен. Такой вариант удобен, когда параметров много и их порядок сложно отследить.

- Если в запросе нужен символ `%` не как плейсхолдер, его указывают как `%%`.

[`Ускорение запросов`](sq03_1.ipynb)

[`Возврат идентификатора`](sq03_2.ipynb)

[`Самостоятельная работа`](sq03_3.ipynb)